In [67]:
from utility.plot_spectrum import plot_spectrum
from utility.adjust_redshift import adjust_redshift
from utility.my_colors import palette_light, pink

from processing.remove_spikes import remove_spikes
from processing.fit_continuum import fit_continuum
from processing.quality_cuts import quality_cuts
from processing.make_bal_mask import make_bal_mask
from processing.fit_gaussians import fit_single_gaussian, fit_two_gaussians, ftest_which_gaussian
from processing.measure_line import measure_line

from sparcl.client import SparclClient
from dl import queryClient as qc
from astropy.table import Table
from astropy.convolution import convolve, Gaussian1DKernel
from scipy.interpolate import interp1d
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import warnings
import json
import os

hamann_fits_path = "data/local/ERQs_core_all_MNRAS.fits"
hamann_coords_path = "data/core-ERQ-stats-coords.csv"
search_log_path = "data/local/hamann-objects-sdss-search-log.json"

### search for objects in sdss

In [24]:
hamann_coords = pd.read_csv(hamann_coords_path, index_col=0)

hamann_coords

,SDSS_id,RA,DEC,ze,i,i-W3,REW,REW_unc,FWHM,FWHM_unc,kt80,NV/CIV,BAL,UV_slope,E(B-V),E(B-V)_unc,FIRST
0,J000610.67+121501.2,1.544491,12.250354,2.31,22.1,8.0,107.0,6.0,4540.0,200.0,0.37,1.88,0,-0.20,0.66,0.01,0.0
1,J000746.19+122223.9,1.942491,12.373315,2.43,21.3,4.9,220.0,6.0,2433.0,71.0,0.28,0.39,0,-0.17,0.19,0.01,0.0
2,J002400.25+245031.9,6.001042,24.842208,2.81,21.6,6.0,144.0,10.0,3523.0,195.0,0.37,2.46,0,-1.14,0.20,0.02,NaN
3,J004713.21+264024.7,11.805059,26.673525,2.56,20.4,4.7,101.0,2.0,3685.0,74.0,0.33,1.92,1,-1.08,0.15,0.01,NaN
4,J005044.95-021217.6,12.687326,-2.204915,2.25,21.5,4.7,147.0,9.0,4343.0,487.0,0.19,0.61,0,-1.73,0.14,0.02,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92,J223754.52+065026.6,339.477177,6.840747,2.61,22.0,5.8,141.0,5.0,1391.0,44.0,0.36,1.03,0,-1.20,0.24,0.02,0.0
93,J225438.30+232714.5,343.659621,23.454052,3.09,22.0,5.5,415.0,20.0,4412.0,146.0,0.36,0.66,0,-0.38,NaN,NaN,NaN
94,J232326.17-010033.1,350.859062,-1.009201,2.36,22.4,7.2,256.0,5.0,3989.0,62.0,0.37,2.16,0,0.91,0.40,0.01,0.0
95,J232828.47+044346.8,352.118656,4.729677,2.56,21.5,5.5,359.0,5.0,1584.0,20.0,0.35,0.37,0,-0.28,0.13,0.02,0.0


In [25]:
def log_result_to_json(sdss_id,best_match_id,results_df, filepath=search_log_path):
    '''Adds best match log entry to specified JSON file.'''

    # create entry
    entry = {
        "timestamp":     datetime.utcnow().isoformat(),
        "SDSS_id":       sdss_id,
        "best_match_id": best_match_id,
        "results_df":    results_df.to_dict(orient="records")
    }

    # open file
    with open(filepath, "r") as f:
        log = json.load(f)

    log.append(entry)  # append entry

    # save to file
    with open(filepath, "w") as f:
        json.dump(log, f, indent=2, default=str)
    
    print(f"Logged entry for {sdss_id}")

In [ ]:
def query_sdss_by_coords(df, z_deviation=0.5, coord_deviaion=0.1):
    '''
    Iterates through dataframe, for each element searches in DESI for the object with smallest difference in RA, DEC, and z (in that order of importance).
    '''

    for elem_idx in range(len(df)):
        # extract ra, dec, z and id for current element
        elem_ra = df["RA"][elem_idx]
        elem_dec = df["DEC"][elem_idx]
        elem_z = df["ze"][elem_idx]
        elem_sdss_id = df["SDSS_id"][elem_idx]

        # query for objects within set devations of current element 
        constraints = {
            "data_release":["BOSS-DR17"],
            "ra":[elem_ra-coord_deviaion,elem_ra+coord_deviaion],
            "dec":[elem_dec-coord_deviaion,elem_dec+coord_deviaion],
            "redshift":[elem_z-z_deviation,elem_z+z_deviation]
            }

        results = client.find(outfields=["specid","ra","dec","redshift"], constraints=constraints)

        results = pd.DataFrame(results.data[1:])

        # if there are results, calculate differences, then sort to get best match
        if len(results) > 0:
            results["z_diff"] = abs(results["redshift"] - elem_z)
            results["ra_diff"] = abs(results["ra"] - elem_ra)
            results["dec_diff"] = abs(results["dec"] - elem_dec)
            results["coord_diff"] = (results["ra"] - elem_ra)**2 + (results["dec"] - elem_dec)**2
            results = results.sort_values(["coord_diff","z_diff"], ascending=[True,True])
            best_match = results.iloc[0,1]

        # if not found, say so
        else:
            best_match = "Not found"            


        # log to json file
        log_result_to_json(elem_sdss_id,best_match,results)

In [84]:
query_sdss_by_coords(hamann_coords)

   redshift               specid        ra        dec        _dr    z_diff  \
2  2.309787  6360287837779417088  1.544491  12.250354  BOSS-DR17  0.000213   
0  2.195029 -5430021594299475968  1.535068  12.250120  BOSS-DR17  0.114971   
4  2.433890  6953634345811990528  1.494368  12.171670  BOSS-DR17  0.123890   
3  2.327830  6953631597032921088  1.471089  12.313582  BOSS-DR17  0.017830   
1  2.328361  6360286738267789312  1.471089  12.313582  BOSS-DR17  0.018361   

        ra_diff  dec_diff    coord_diff  
2  2.000000e-07  0.000000  4.000000e-14  
0  9.423300e-03  0.000234  8.885334e-05  
4  5.012300e-02  0.078684  8.703487e-03  
3  7.340220e-02  0.063228  9.385663e-03  
1  7.340220e-02  0.063228  9.385663e-03  
6360287837779417088
Logged entry for J000610.67+121501.2
   redshift               specid        ra        dec        _dr    z_diff  \
0  2.433351  6360267496814303232  1.942491  12.373315  BOSS-DR17  0.003351   
3  2.433563  6953611256067807232  1.942491  12.373315  BOSS-DR17  

ConnectTimeout: HTTPSConnectionPool(host='astrosparcl.datalab.noirlab.edu', port=443): Max retries exceeded with url: /api/find/?limit=500 (Caused by ConnectTimeoutError(<HTTPSConnection(host='astrosparcl.datalab.noirlab.edu', port=443) at 0x7fd00b176590>, 'Connection to astrosparcl.datalab.noirlab.edu timed out. (connect timeout=1.1)'))

### actual line measurements

In [ ]:
# hamann_data = Table.read("data/local/ERQs_core_all_MNRAS.fits").to_pandas() #pd.read_csv(hamann_erq_path,index_col=0)

# hamann_data["sdss_name"] = hamann_data["sdss_name"].astype("string")

# match_df = pd.read_json(search_log_path)
# match_df = match_df[["SDSS_id","best_match_id"]].copy()
# match_df["SDSS_id"] = match_df["SDSS_id"].str[1:]

# match_df = match_df.rename(columns={"best_match_id":"targetid", "SDSS_id":"sdss_name"})

# hamann_erq = hamann_data.merge(match_df,on="sdss_name",how="left")

# hamann_erq.info()